# M2 v2 — Entrenamiento con aumento uniforme

**TFM · HAM10000** · Experimento `m2-entrenamiento-v2` · Etiqueta git `m2-v2`

Una semilla y las arquitecturas de `ARCHS`. El código está en `src/`; este cuaderno solo comprueba las huellas y llama a `lanzar_run`.

Piloto (Alex): `ARCHS = ["densenet121"]` y `SEED = 42`. Si termina con `estado=final` y `alarma_atajo=false`, cada cuenta corre las tres arquitecturas con su semilla. Alex completa después ResNet50 y EfficientNetB0 con la 42: el DenseNet121 ya cerrado se salta.

Si Colab se desconecta, volver a ejecutar todas las celdas. Una arquitectura con `estado=final` no se repite. Los checkpoints quedan en Drive, `checkpoints-v2/m2-<arch>-s<semilla>/`, incluido el mejor de la fase 1.

No evalúa el conjunto de prueba y no mueve alias.


## 0. Parámetros

La semilla y la lista de arquitecturas se eligen solo aquí.

In [ ]:
TAG = "m2-v2"
AUTOR = "alex"          # alex | adrian | christian
SEED = 42               # Alex 42, Adrián 123, Christian 2024
ARCHS = ["resnet50", "densenet121", "efficientnetb0"]
# Piloto de Alex: ARCHS = ["densenet121"]

REVISION_NV = False     # Adrián: True para la figura de 45 variantes de nv


## 1. Sesión

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "Sin GPU"
import sys
print("Python", sys.version.split()[0])
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os
OWNER, REPO = "aReinoso007", "tfm-melanoma-cnn-gradcam"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/{OWNER}/{REPO}.git {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q --tags origin
!git checkout -q {TAG}
!git rev-parse HEAD
!git status --porcelain
!pip install -q -r requirements.txt


## 2. Huellas

Si alguna no coincide, esta celda para y no se entrena. El zip se acepta cuando su contenido coincide con las huellas publicadas: 10 015 imágenes y el SHA-256 de `HAM10000_metadata.csv`. La huella del archivo zip se imprime para compararla entre cuentas; ese archivo solo vive en Drive y no tiene una constante publicada.


In [ ]:
import os, shutil, zipfile, time
from pathlib import Path
from src.tracking import load_config, file_sha256

HUELLAS = {
    "split_v1.csv": "98485423d758cae1ed8b7adf64c3e92d31a6afd4d48de7a26e89355238e34b38",
    "class_weights.json": "965f0301fdfe13fcc3925f1263c2171cd25f49760408261fc39b83b337b04d86",
    "HAM10000_metadata.csv": "06aee5e603d313ba69de38263053c792a1dff030eb9c44a96ceb477ceefb99a5",
    "src/augmentation.py": "e6649f772196e4090641430e829a0475f72ce5ffcf4d4f0937bf33de4046d72e",
    "configs/base.yaml": "871fdc3d5784dbd7546339d94c0a6fd8d0766fb1ed7e800c093f8d051955de12",
}
N_IMAGENES = 10015

cfg = load_config("configs/base.yaml")
assert cfg["config_version"] == 2, cfg.get("config_version")
assert cfg["data"]["augmentation"]["politica"] == "uniforme"

zip_path = cfg["paths"]["dataset_zip"]
assert Path(zip_path).is_file(), f"No está el zip: {zip_path}"
print("huella zip", file_sha256(zip_path))

LOCAL = Path(cfg["paths"]["local_data_dir"])
if not (LOCAL / "images").exists():
    t0 = time.time()
    shutil.copy(zip_path, "/content/HAM10000.zip")
    with zipfile.ZipFile("/content/HAM10000.zip") as z:
        z.extractall(LOCAL)
    os.remove("/content/HAM10000.zip")
    print(f"Descomprimido en {LOCAL} ({time.time() - t0:.0f} s)")

comprobaciones = {
    "split_v1.csv": "data/splits/split_v1.csv",
    "class_weights.json": "data/splits/class_weights.json",
    "HAM10000_metadata.csv": cfg["paths"]["metadata_csv"],
    "src/augmentation.py": "src/augmentation.py",
    "configs/base.yaml": "configs/base.yaml",
}
fallos = []
for nombre, ruta in comprobaciones.items():
    huella = file_sha256(ruta)
    if huella != HUELLAS[nombre]:
        fallos.append(f"{nombre}: {huella} != {HUELLAS[nombre]}")
n = len(list((LOCAL / "images").glob("*.jpg")))
if n != N_IMAGENES:
    fallos.append(f"imágenes del zip: {n} != {N_IMAGENES}")
if fallos:
    raise SystemExit("Huella distinta. No se entrena.\n" + "\n".join(fallos))
print("[+] Huellas coinciden. Se puede entrenar.")


In [ ]:
import os
import mlflow
from google.colab import userdata
from src.tracking import setup_mlflow, git_dirty

os.environ["MLFLOW_TRACKING_USERNAME"] = userdata.get("DAGSHUB_USER")
os.environ["MLFLOW_TRACKING_PASSWORD"] = userdata.get("DAGSHUB_TOKEN")
print("MLflow:", setup_mlflow(cfg))
print("git_dirty =", git_dirty(), "| tag", TAG, "| semilla", SEED, "|", ARCHS)
assert not git_dirty(), "El clon de la etiqueta tiene cambios. No se entrena."
assert SEED in cfg["training"]["seeds"], SEED
assert set(ARCHS) <= set(cfg["training"]["architectures"])


## 3. Entrenar

Cada arquitectura es un run `m2-<arch>-s<semilla>` en `m2-entrenamiento-v2`. Al terminar registra la prueba de regresión del atajo. La alarma salta si la fracción predicha nv baja más de 0,20 al forzar el aumento (semillas 11, 22 y 33). El run queda en `estado=final` igual; si la alarma salta, el piloto no pasa.


In [ ]:
from pathlib import Path
from src.training import lanzar_run

def run_cerrado(arch):
    id_file = Path(cfg["paths"]["checkpoints_v2_dir"]) / f"m2-{arch}-s{SEED}" / "mlflow_run_id.txt"
    if not id_file.exists():
        return None
    run = mlflow.get_run(id_file.read_text().strip())
    if run.info.status == "FINISHED" and run.data.tags.get("estado") == "final":
        return run
    return None

for arch in ARCHS:
    previo = run_cerrado(arch)
    if previo is not None:
        print(f"{arch}: ya está en estado=final ({previo.info.run_id}). No se repite.")
        continue
    print(f"\n{'=' * 20} {arch} · semilla {SEED} {'=' * 20}")
    try:
        lanzar_run(arch, SEED, cfg, "real", AUTOR)
    except RuntimeError as e:
        if "ya terminó" not in str(e):
            raise
        cerrado = run_cerrado(arch)
        if cerrado is None:
            raise
        print(f"{arch}: el run ya había terminado ({cerrado.info.run_id}).")


## 4. Resumen copiable

In [ ]:
from pathlib import Path

def _fmt(m, clave):
    v = m.get(clave)
    return "—" if v is None else f"{v:.3f}"

lineas = []
alguna_alarma = False
for arch in ARCHS:
    id_file = Path(cfg["paths"]["checkpoints_v2_dir"]) / f"m2-{arch}-s{SEED}" / "mlflow_run_id.txt"
    if not id_file.exists():
        lineas.append(f"{arch}  pendiente")
        continue
    run = mlflow.get_run(id_file.read_text().strip())
    m, t = run.data.metrics, run.data.tags
    minutos = m.get("tiempo_total_s")
    minutos_txt = "—" if minutos is None else f"{minutos / 60:.1f}"
    alarma = t.get("alarma_atajo", "—")
    alguna_alarma = alguna_alarma or alarma == "true"
    lineas.append(
        f"{arch}  run={run.info.run_id}  estado={t.get('estado', '—')}  "
        f"macro_f1={_fmt(m, 'val_macro_f1')}  sens_mel={_fmt(m, 'val_mel_sensitivity')}  "
        f"auc_mel={_fmt(m, 'val_auc_mel')}  "
        f"frac_nv_limpio={_fmt(m, 'val_frac_nv_limpio')}  "
        f"frac_nv_aumentado={_fmt(m, 'val_frac_nv_aumentado')}  "
        f"minutos={minutos_txt}  alarma_atajo={alarma}"
    )

print(f"semilla {SEED}  autor {AUTOR}  experimento m2-entrenamiento-v2")
print("\n".join(lineas))
if alguna_alarma:
    print("ALARMA de atajo. El piloto no pasa: no lances el resto de runs.")
else:
    print("Sin alarma de atajo en las arquitecturas ya cerradas.")


## 5. Revisión visual de nv (solo Adrián)

Poner `REVISION_NV = True` en la celda de parámetros y ejecutar esta celda. Genera 5 semillas (42, 1, 2, 3 y 100) × 9 variantes = 45, con el aumento completo, y guarda la figura en Drive.

Criterios, los mismos de la revisión original: deja de parecer la misma lesión, corta el borde de la lesión, cambia el color de forma visible, o muestra esquinas raras por la rotación.


In [ ]:
if not REVISION_NV:
    print("REVISION_NV = False. Esta celda no genera la figura.")
else:
    import tensorflow as tf
    import matplotlib.pyplot as plt
    import pandas as pd
    from src.augmentation import augmentar
    from src.data import _cargar, split_df

    SEMILLAS = (42, 1, 2, 3, 100)
    N_VAR = 9
    train = split_df("train")
    nv = train[train.dx == "nv"]
    images = Path(cfg["paths"]["images_dir"])
    fig, axes = plt.subplots(len(SEMILLAS), N_VAR + 1, figsize=(1.35 * (N_VAR + 1), 1.45 * len(SEMILLAS)))
    usadas = {}
    for r, semilla in enumerate(SEMILLAS):
        fila = nv.sample(1, random_state=semilla).iloc[0]
        usadas[semilla] = fila.image_id
        base = tf.cast(_cargar(str(images / f"{fila.image_id}.jpg")), tf.float32)
        variantes = [base] + [
            augmentar(base, 5, tf.constant([semilla, j], tf.int64)) for j in range(N_VAR)
        ]
        for j, v in enumerate(variantes):
            ax = axes[r, j]
            ax.imshow(tf.cast(tf.clip_by_value(v, 0, 255), tf.uint8).numpy())
            ax.set_xticks([]); ax.set_yticks([])
        axes[r, 0].set_ylabel(f"s{semilla}", rotation=0, ha="right", va="center")
    axes[0, 0].set_title("original")
    fig.suptitle("nv · aumento completo · 5 semillas × 9 variantes", fontsize=11)
    fig.subplots_adjust(wspace=0.04, hspace=0.08, top=0.92)
    destino = Path(cfg["paths"]["drive_root"]) / "revision-nv-v2.png"
    fig.savefig(destino, dpi=200, bbox_inches="tight", facecolor="white")
    plt.show()
    print("Figura:", destino)
    print("Imágenes:", usadas)
